# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR<sup>2</sup>](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, referencing all dataset entities by their `@id` attribute.

### Dataset Source
The Croissant schema source URL for this dataset is:

[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset name and description (accessed as attributes)
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Explore the available record sets and their associated fields and IDs.

In [ ]:
# List all record sets with their '@id' and major fields/columns
print("Available record sets (by @id):\n")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- Record set @id: {rs.id}")
    print(f"  Name: {rs.name if hasattr(rs,'name') else ''}")
    print(f"  Description: {rs.description if hasattr(rs,'description') else ''}")
    print(f"  Fields:")
    # List associated fields of this record set
    for field in rs.fields:
        print(f"    - Field @id: {field.id} | Name: {field.name if hasattr(field,'name') else ''} | Data type: {field.data_type if hasattr(field,'data_type') else ''}")
    print("-")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. **All entities use their `@id` field for reference.**

> To customize this step, choose the desired `@id` for a record set from the list above.


In [ ]:
# List all record set @id values
record_set_ids = [rs.id for rs in dataset.record_sets]
print("All available record set @ids:")
for rid in record_set_ids:
    print("-", rid)

# Choose a main record set (by @id); update if you wish to explore another
# For this dataset, there's typically just one tabular data record set--use the first one.
main_record_set_id = record_set_ids[0] if record_set_ids else None
print(f"\nUsing main record set @id: {main_record_set_id}\n")

# Extract data for all record sets
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

if main_record_set_id is not None:
    print("Columns for the main record set:")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print("No record sets with tabular data found.")

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps using fields referenced **by their `@id`**. We'll illustrate filtering, normalization, and aggregation/grouping. Update the field `@id`s if you'd like to analyze other columns.


In [ ]:
# Choose a numeric field (by @id) for analysis from the field list above
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]

    # Find all numeric fields (typically float/int); pick first found
    target_numeric_field_id = None
    rs_obj = next(rs for rs in dataset.record_sets if rs.id==main_record_set_id)
    for field in rs_obj.fields:
        if hasattr(field, 'data_type') and (field.data_type in ['schema:Number', 'schema:Float', 'schema:Integer', 'Float', 'Number', 'Integer']):
            if field.id in df.columns:
                target_numeric_field_id = field.id
                break
    
    print(f"Numeric field selected (by @id): {target_numeric_field_id}")

    # Filter: keep rows with numeric_field > threshold
    threshold = 10
    if target_numeric_field_id:
        filtered_df = df[pd.to_numeric(df[target_numeric_field_id], errors='coerce') > threshold].copy()
        print(f"Filtered records with {target_numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{target_numeric_field_id}_normalized"] = (filtered_df[target_numeric_field_id].astype(float) - filtered_df[target_numeric_field_id].astype(float).mean()) / filtered_df[target_numeric_field_id].astype(float).std()
        print(f"\nNormalized {target_numeric_field_id} for filtered records:")
        print(filtered_df[[target_numeric_field_id, f"{target_numeric_field_id}_normalized"]].head())

        # Try to find a categorical/group field (string type, not the numeric field)
        group_field_id = None
        for field in rs_obj.fields:
            if hasattr(field, 'data_type') and (field.data_type in ['schema:Text', 'Text', 'schema:String', 'String']):
                if field.id in df.columns and field.id != target_numeric_field_id:
                    group_field_id = field.id
                    break
        if group_field_id:
            print(f"\nGrouping by field (by @id): {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print("Grouped means:")
            print(grouped_df.head())
        else:
            print("No categorical/text field for grouping found in data.")
    else:
        print("No numeric field available for analysis.")
else:
    print("No main DataFrame available for EDA.")

## 5. Visualization

Visualize distributions or relationships using fields referenced by their `@id`. Here, we create a histogram of the selected numeric field and, if available, a boxplot grouped by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and target_numeric_field_id is not None:
    # Histogram for the normalized numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[f"{target_numeric_field_id}_normalized"], bins=30, kde=True)
    plt.title(f"Distribution of normalized {target_numeric_field_id}")
    plt.xlabel(f"{target_numeric_field_id}_normalized")
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group field if available
    if 'group_field_id' in locals() and group_field_id is not None:
        plt.figure(figsize=(9,4))
        sns.boxplot(data=filtered_df, x=group_field_id, y=f"{target_numeric_field_id}_normalized")
        plt.xticks(rotation=45)
        plt.title(f"{target_numeric_field_id} (normalized) grouped by {group_field_id}")
        plt.show()

## 6. Conclusion

- We loaded the [FAIR<sup>2</sup>](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset package using the Croissant schema and explored its record sets, referencing all entities by their `@id` field as per best practice.
- Tabular data was loaded and inspected; a main numeric field was filtered, normalized, and (when present) grouped by a categorical field for analysis.
- Distributional statistics and basic visualizations help reveal patterns and outliers.

**Next steps:** Use the rich field metadata (`@id`, `description`, `dataType`) to drive advanced feature engineering, join multiple record sets, or extend this workflow for custom downstream analyses.